# Running a P300 speller from a notebook

The same experiment the graphical launcher runs, driven from python — useful
when you want to change something, look at the data as it arrives, or analyse
a recording afterwards.

You need `numpy` and `matplotlib`; everything else is in the package. Run the
cells in order.

> For a session with a real participant, use the manual in `docs/MANUAL.md` and
> the graphical launcher (`python -m pyspeller run`). This notebook is for
> learning the system and for analysis.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))     # run from the docs/ directory

import numpy as np
import matplotlib.pyplot as plt

from pyspeller import SpellerConfig
from pyspeller.experiment import LocalExperiment

## 1. Configure the session

`SpellerConfig` is the single place where the experiment is described: the
matrix, the timing, the filter band, the montage. `speed` compresses
experiment time — at `speed=10` a two-minute calibration takes twelve seconds,
and the data in the buffer is identical, because everything is indexed by
samples rather than seconds.

In [ ]:
config = SpellerConfig(
    port=0,                       # 0 = pick a free port
    speed=10.0,                   # run 10x faster than real time
    n_repetitions=12,             # flashes of each row/column per letter
    calibration_letters=tuple('BRAIN'),
    feedback_letters=tuple('BCI'),
)
print('matrix %dx%d, %d channels at %g Hz'
      % (config.n_rows, config.n_cols, config.n_channels, config.fsample))
print('\n'.join(' '.join(row) for row in config.symbols))

## 2. Start the system

`LocalExperiment` starts everything a session needs in this process: the buffer
server, a simulated amplifier, the stimulus client and the signal processing.
With real hardware you would instead start `python -m pyspeller buffer` and
`python -m pyspeller lsl` in terminals and connect a `BufferClient` to them —
see the last section.

In [ ]:
experiment = LocalExperiment(config, verbose=True).start()
client = experiment.processor.client          # our window onto the buffer
print('buffer on port %d, %d samples so far' % (config.port, client.poll()[0]))

## 3. Look at the raw signal

Always look at the data before trusting anything downstream.

In [ ]:
import time
time.sleep(2.0)                                # let some data accumulate
nsamples = client.poll()[0]
data = client.get_data(max(0, nsamples - 512), nsamples - 1)   # [samples x channels]

fig, ax = plt.subplots(figsize=(10, 4))
offset = 40
for i, label in enumerate(config.channels):
    ax.plot(np.arange(data.shape[0]) / config.fsample, data[:, i] - i * offset, lw=0.7)
    ax.text(-0.15, -i * offset, label, ha='right', va='center')
ax.set_xlabel('seconds'); ax.set_yticks([]); ax.set_title('raw EEG from the buffer')
plt.show()

## 4. Calibration

The stimulus client cues each letter and flashes rows and columns; the signal
processing client cuts a 600 ms epoch after every flash and labels it with
whether the flashed group contained the cued letter.

In [ ]:
epochs, labels = experiment.calibrate()
print('%d epochs of shape %s, %d targets'
      % (len(labels), epochs.shape[1:], labels.sum()))

## 5. The P300 itself

Average the target and non-target epochs. A P3b should be visible over the
centro-parietal electrodes, peaking around 300 ms after the flash.

In [ ]:
t = np.arange(epochs.shape[2]) / config.fsample * 1000      # ms
target = epochs[labels == 1].mean(axis=0)
nontarget = epochs[labels == 0].mean(axis=0)

fig, axes = plt.subplots(2, 4, figsize=(13, 5), sharex=True, sharey=True)
for i, (ax, label) in enumerate(zip(axes.ravel(), config.channels)):
    ax.plot(t, nontarget[i], label='non-target', lw=1)
    ax.plot(t, target[i], label='target', lw=1.5)
    ax.axvline(300, color='0.8', lw=0.8, zorder=0)
    ax.set_title(label)
axes[0, 0].legend(fontsize=8)
fig.supxlabel('ms after the flash'); fig.supylabel('microvolts')
plt.tight_layout(); plt.show()

## 6. Train the classifier

The pipeline is detrend → common average reference → 0.5–10 Hz band-pass →
downsample to 16 Hz → shrinkage LDA over channels × time. The report is a
stratified 5-fold cross-validated AUC: above ~0.75 is a good session, below
0.65 means something is wrong (see the troubleshooting table in the manual).

In [ ]:
report = experiment.train()
report

What did the classifier learn? The weights have one value per channel and time
point, so they can be shown as an image — the positive band around 300 ms over
the parietal channels is the P300 the classifier is keying on.

In [ ]:
clf = experiment.processor.classifier
weights = clf.lda.weights.reshape(clf.feature_shape)        # channels x time
times = np.linspace(0, config.trlen_ms, weights.shape[1])

fig, ax = plt.subplots(figsize=(9, 3.5))
limit = np.abs(weights).max()
image = ax.imshow(weights, aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit,
                  extent=[times[0], times[-1], len(config.channels) - 0.5, -0.5])
ax.set_yticks(range(len(config.channels))); ax.set_yticklabels(config.channels)
ax.set_xlabel('ms after the flash'); ax.set_title('classifier weights')
fig.colorbar(image, ax=ax); plt.show()

## 7. Spelling

In feedback the classifier scores every flash, the scores are averaged per row
and per column, and the best row and column intersect at the decoded letter.

In [ ]:
pairs = experiment.feedback()
for cued, decoded in pairs:
    print('cued %s -> decoded %s %s' % (cued, decoded, '' if cued == decoded else '  <-- wrong'))
print('typed: %r' % experiment.stimulus.spelled)

Corrections work the same way they do in the window: `DEL` is a cell in the
matrix, and any client can send the same edit as an event.

In [ ]:
from pyspeller.speller import text as speller_text

print('before:', repr(experiment.stimulus.spelled))
experiment.stimulus.apply_edit(speller_text.DELETE)
print('after :', repr(experiment.stimulus.spelled))

In [ ]:
experiment.stop()          # always stop: it closes the buffer and the amplifier

## 8. Analysing a recording afterwards

A session recorded with `--save` (or `python -m pyspeller save`) is a directory
of four files in the FieldTrip offline-buffer format. Point this at one of your
own recordings.

In [ ]:
from pyspeller.acquisition.saver import load_session, load_epochs

# path = os.path.expanduser('~/output/speller/S01/260921/1503/raw_buffer')
# header, samples, events = load_session(path)
# print(header, samples.shape)
# print([e.value for e in events if e.type == 'classifier.prediction'])
# epochs, labels, meta = load_epochs(os.path.join(path, 'calibration_epochs.npz'))

## 9. The same against a g.tec amplifier

Start the g.tec LSL connector, then in two terminals:

```bash
python -m pyspeller buffer
python -m pyspeller lsl --name "g.USBamp-UB-2016.03.06"
```

and connect to that buffer instead of starting a simulator:

```python
from pyspeller.buffer import BufferClient
from pyspeller.speller.sigproc import SignalProcessor

client = BufferClient('localhost', 1972).connect()
client.wait_for_header()                 # channels and rate come from the amplifier
processor = SignalProcessor(client, config, save_dir='~/output/S01')
processor.run_phase_loop()               # obeys the buttons on the control panel
```

Everything above — the plots, the training report, the weights — works the same
on the real data.